# 04. Evaluation — Confusion Matrix & Accuracy
# 04. 평가 — 혼동 행렬 & 정확도

**Grayspot Detection Pipeline · Stage 1 Prototype**

This notebook evaluates the trained Grayspot model by computing:
- Overall Accuracy / Per-color Accuracy
- Per-class Precision, Recall, F1, MAE
- 6×6 Confusion Matrix (Plotly interactive HTML)
- Misclassified sample gallery (Plotly HTML)
- Confidence distribution (Plotly HTML)

이 노트북은 학습된 Grayspot 모델을 다음 항목으로 평가합니다:
- 전체 정확도 / 색상별 정확도
- 클래스별 Precision, Recall, F1, MAE
- 6×6 혼동 행렬 (Plotly 인터랙티브 HTML)
- 오분류 샘플 갤러리 (Plotly HTML)
- 신뢰도 분포 (Plotly HTML)

---
**Prerequisites / 선행 조건**
```
pip install torch torchvision scikit-learn plotly pandas numpy Pillow
```
**Runs on / 실행 환경**: Windows & macOS (CPU / CUDA / MPS)

---
### ⚠️ plt.show() / fig.show() 제거 안내
VS Code 내장 Jupyter (특히 macOS) 에서 `plt.show()` 와 Plotly `fig.show()` 가
오류를 유발합니다. 이 노트북에서는 두 메서드를 **완전히 제거**하고 대신:
- 모든 차트 → HTML 파일 저장 후 `open_in_browser()` 로 기본 브라우저에서 열기

VS Code built-in Jupyter (especially macOS) raises errors for `plt.show()` and
Plotly `fig.show()`. This notebook removes both entirely:
- All charts → saved as HTML, opened via `open_in_browser()`

---
### 📁 Project Directory Layout / 프로젝트 디렉토리 구조
```
CMYK_MAIN/                          ← ROOT_DIR  (notebooks/ 기준 ../../)
└── data_set/
    ├── labeled/                    ← LABELED_DIR  (06번과 동일 / Same as 06)
    │   ├── C/{0..5}/*.png
    │   ├── K/{0..5}/*.png
    │   ├── M/{0..5}/*.png
    │   └── Y/{0..5}/*.png
    ├── models/                     ← 체크포인트 .pt 파일
    │   ├── baseline_C.pt
    │   ├── baseline_Y.pt
    │   └── phase0_backbone_C.pt
    ├── reports/                    ← OUTPUT_DIR  (이 노트북 출력)
    │   ├── logs/
    │   ├── baseline_history_C.csv
    │   └── baseline_history_Y.csv
    ├── embedding_viz/              ← 06_embedding_viz.ipynb 출력
    └── labels_v0.csv               ← LABELS_CSV  (06번과 공유)
```


---
## Cell 0 · 환경 확인 / Environment Check

In [ ]:
# 필요 패키지 설치 (최초 1회) / Install required packages (first time only)
# !pip install torch torchvision scikit-learn plotly pandas numpy Pillow

In [1]:
# ─────────────────────────────────────────────────────────────────────
# 0-A. Python 3.11.5 기준 임포트 / Python 3.11.5 standard imports
# matplotlib 미사용 — Plotly 단독으로 모든 시각화 처리
# No matplotlib — Plotly handles all visualization (same policy as 06_embedding_viz)
# ─────────────────────────────────────────────────────────────────────
import os
import sys
import json
import random
import warnings
import webbrowser                              # HTML 파일 브라우저 열기 / Open HTML in browser
from pathlib import Path
from typing import Optional, Dict, List, Tuple

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ── Python 버전 확인 / Python version guard ───────────────────────────
assert sys.version_info[:2] == (3, 11), (
    f'Python 3.11.x required, but found {sys.version}. '
    f'Python 3.11.x 가 필요합니다. 현재 버전: {sys.version}'
)

print('Python 3.11.5 기준 노트북 / Python 3.11.5 standard notebook')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA 사용 가능 / Available: {torch.cuda.is_available()}')

Python 3.11.5 기준 노트북 / Python 3.11.5 standard notebook
PyTorch  : 2.2.2+cu118
CUDA 사용 가능 / Available: True


In [2]:
# ─────────────────────────────────────────────────────────────────────
# 0-B. open_in_browser 유틸리티 (06_embedding_viz 와 동일 구현)
# 0-B. open_in_browser utility  (identical implementation to 06_embedding_viz)
# ─────────────────────────────────────────────────────────────────────
def open_in_browser(path) -> None:
    """
    저장된 HTML 파일을 기본 브라우저로 연다.
    Opens the saved HTML file in the default browser.

    Mac / Linux / Windows 모두 file:// URI 방식을 사용하여 안정적으로 동작한다.
    Uses file:// URI on Mac / Linux / Windows for reliable cross-platform behavior.

    webbrowser.open()에 일반 경로 문자열을 넘기면 macOS에서 열리지 않을 수 있으므로
    Path.as_uri()로 file:// 형태로 변환한다.
    Passing a plain path string to webbrowser.open() may fail on macOS,
    so Path.as_uri() is used to convert it to file:// format.
    """
    uri = Path(path).resolve().as_uri()   # 절대경로 → file:// URI
    webbrowser.open(uri)


# ── 디바이스 자동 감지 / Auto-select compute device ──────────────────
# Windows: CUDA → CPU  /  macOS: MPS(Apple Silicon) → CPU
def get_device() -> torch.device:
    """컴퓨팅 디바이스를 자동으로 선택합니다. / Auto-select compute device."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')    # Apple Silicon / 애플 실리콘
    return torch.device('cpu')

DEVICE = get_device()
print(f'⚙️  Compute device / 연산 디바이스 : {DEVICE}')

# ── 전역 랜덤 시드 고정 (재현성) / Fix global random seed ─────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'🎲  Random seed fixed / 랜덤 시드 고정: {SEED}')

⚙️  Compute device / 연산 디바이스 : cuda
🎲  Random seed fixed / 랜덤 시드 고정: 42


---
## Cell 1 · 경로 및 설정값 / Path & Configuration

> **이 셀만 수정**하면 모든 경로가 자동으로 반영됩니다.  
> **Edit only this cell** — all paths propagate automatically.

In [3]:
# ── 경로 설정 / Path settings ─────────────────────────────────────────
# 06_embedding_viz.ipynb 와 완전히 동일한 기준점 사용
# Identical anchor to 06_embedding_viz.ipynb

ROOT_DIR    = Path('../..').resolve()                   # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / 'data_set' / 'labeled'        # 라벨링 폴더 (06번과 동일)
LABELS_CSV  = ROOT_DIR / 'data_set' / 'labels_v0.csv' # 라벨 CSV   (06번과 공유)
MODELS_DIR  = ROOT_DIR / 'data_set' / 'models'         # 체크포인트 폴더
#OUTPUT_DIR  = ROOT_DIR / 'data_set' / 'reports'        # 이 노트북 전용 출력 폴더
OUTPUT_DIR = ROOT_DIR / 'data_set' / 'embedding_viz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 체크포인트 설정 / Checkpoint settings ─────────────────────────────
# Phase 2 체크포인트 지정. None → 랜덤 가중치로 실행 (프로토타입 모드)
# Set Phase 2 checkpoint. None → random weights (prototype mode)
#
# 사용 예 / Examples:
#   CHECKPOINT = MODELS_DIR / 'baseline_C.pt'
#   CHECKPOINT = MODELS_DIR / 'baseline_Y.pt'
CHECKPOINT = None

# ── 데이터 설정 / Data settings ───────────────────────────────────────
# 06_embedding_viz.ipynb 와 동일한 채널 / 컬럼 규칙
# Same channel / column convention as 06_embedding_viz.ipynb
CHANNELS      = ['Y', 'M', 'C', 'K']    # 처리 순서 / Processing order
NUM_LEVELS    = 6                         # Level 0 ~ 5
IMAGE_SIZE    = 224                       # 모델 입력 크기 / Model input size
BATCH_SIZE    = 32

# CSV 컬럼명 매핑 (06번과 동일) / CSV column mapping (same as 06)
# labels_v0.csv 컬럼 형식: filename, C, M, Y, K
COLOR_COLUMNS = {'Y': 'Y', 'M': 'M', 'C': 'C', 'K': 'K'}

# ── 모델 설정 / Model settings ────────────────────────────────────────
BACKBONE_NAME = 'efficientnet_b0'    # 'efficientnet_b0' | 'resnet18' | 'resnet34'

# ── 성능 목표 / Performance targets (PRD §1.4) ────────────────────────
TARGET_OVERALL_ACC   = 0.90
TARGET_PER_CLASS_F1  = 0.80
TARGET_PER_COLOR_ACC = 0.85
TARGET_MAE           = 0.50

# ── PRD §14.2 신뢰도 임계값 / Confidence thresholds ──────────────────
CONF_THRESH_AUTO   = 0.8    # 자동 판정 / Auto judgment
CONF_THRESH_WARN   = 0.5    # 경고 포함 자동 / Warn + auto
CONF_THRESH_MANUAL = 0.3    # 수동 검수 대기 / Manual queue

# ── 시각화 색상 (06번과 동일) / Visualization colors (same as 06) ────
LEVEL_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#9b59b6', '#1a1a2e']
CMYK_COLORS  = {'Y': '#f5e642', 'M': '#e91e8c', 'C': '#00b4d8', 'K': '#444444'}

# ── 확인 출력 / Verification output ──────────────────────────────────
print(f'ROOT_DIR    : {ROOT_DIR}')
print(f'LABELED_DIR : {LABELED_DIR}')
print(f'LABELS_CSV  : {LABELS_CSV}')
print(f'MODELS_DIR  : {MODELS_DIR}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')
print(f'CHECKPOINT  : {CHECKPOINT}')
print(f'CSV exists / 파일 존재: {LABELS_CSV.exists()}')
print()

print('Labeled folder counts / 라벨링 폴더 샘플 수:')
for ch in CHANNELS:
    for lv in range(NUM_LEVELS):
        folder = LABELED_DIR / ch / str(lv)
        count  = len(list(folder.glob('*'))) if folder.exists() else 0
        print(f'  [{ch}] Level {lv}: {count}개')

ROOT_DIR    : C:\visualcode\CMYK_jin_clone\CMYK_MAIN
LABELED_DIR : C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\labeled
LABELS_CSV  : C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\labels_v0.csv
MODELS_DIR  : C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\models
OUTPUT_DIR  : C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz
CHECKPOINT  : None
CSV exists / 파일 존재: False

Labeled folder counts / 라벨링 폴더 샘플 수:
  [Y] Level 0: 0개
  [Y] Level 1: 0개
  [Y] Level 2: 0개
  [Y] Level 3: 0개
  [Y] Level 4: 0개
  [Y] Level 5: 0개
  [M] Level 0: 0개
  [M] Level 1: 0개
  [M] Level 2: 0개
  [M] Level 3: 0개
  [M] Level 4: 0개
  [M] Level 5: 0개
  [C] Level 0: 0개
  [C] Level 1: 0개
  [C] Level 2: 0개
  [C] Level 3: 0개
  [C] Level 4: 0개
  [C] Level 5: 0개
  [K] Level 0: 0개
  [K] Level 1: 0개
  [K] Level 2: 0개
  [K] Level 3: 0개
  [K] Level 4: 0개
  [K] Level 5: 0개


---
## Cell 2 · 라벨 로드 / Label Loading

06번과 동일한 `load_labels()` / `create_dummy_labels()` 패턴  
Same `load_labels()` / `create_dummy_labels()` pattern as 06_embedding_viz

In [4]:
def load_labels(labels_csv: Path, color_columns: dict) -> pd.DataFrame:
    """
    라벨 CSV를 로드하여 long-format DataFrame으로 변환한다.
    Loads the label CSV and converts it to a long-format DataFrame.

    원본 CSV 형식 (wide) / Original CSV format (wide):
        filename, C, M, Y, K

    반환 형식 (long) / Return format (long):
        filename, color, level, confidence
        - 각 이미지당 4개 행 (Y/M/C/K) / 4 rows per image
    """
    df = pd.read_csv(labels_csv)
    print(f'[CSV 로드 / Loaded] 원본 행 수 / Rows: {len(df)}, 컬럼 / Columns: {list(df.columns)}')

    # Wide → Long 변환 / Wide to Long conversion
    records = []
    for _, row in df.iterrows():
        for color_code, col_name in color_columns.items():
            if col_name not in df.columns:
                continue    # 컬럼 없으면 건너뜀 / Skip if column not found
            records.append({
                'filename'  : row['filename'],
                'color'     : color_code,
                'level'     : int(row[col_name]),
                'confidence': row.get('confidence', '확실'),
            })

    long_df = pd.DataFrame(records)
    print(f'[변환 완료 / Converted] Long-format 행 수 / Rows: {len(long_df)}')
    if len(long_df) > 0:
        print(long_df.groupby(['color', 'level']).size().unstack(fill_value=0))
    return long_df


def create_dummy_labels(n_per_class: int = 20) -> pd.DataFrame:
    """
    S1 프로토타입: 실제 라벨 CSV가 없을 때 사용하는 더미 데이터.
    S1 Prototype: Dummy data used when real label CSV is not available.
    Level 0~5, 색상 Y/M/C/K 조합으로 균일하게 생성.
    Generated uniformly for Level 0~5 and color Y/M/C/K combinations.
    """
    print('[더미 데이터 / Dummy] 실제 CSV 미존재 → 더미 데이터 생성 / CSV not found → generating dummy data')
    records = []
    idx     = 0
    for c in CHANNELS:
        for lv in range(NUM_LEVELS):
            for _ in range(n_per_class):
                records.append({
                    'filename'  : f'scan_dummy_{idx:04d}.png',
                    'color'     : c,
                    'level'     : lv,
                    'confidence': '확실' if np.random.rand() > 0.2 else '불확실',
                })
                idx += 1
    return pd.DataFrame(records)


# 실제 CSV 존재 여부에 따라 분기 / Branch based on CSV existence
if LABELS_CSV.exists():
    df_labels = load_labels(LABELS_CSV, COLOR_COLUMNS)
else:
    df_labels = create_dummy_labels(n_per_class=20)

# level_str 컬럼 추가 (Plotly 색상 매핑용) / Add for Plotly color mapping
df_labels['level_str'] = df_labels['level'].astype(str)

print(f'\n총 샘플 수 / Total samples: {len(df_labels)}')
df_labels.head()

[더미 데이터 / Dummy] 실제 CSV 미존재 → 더미 데이터 생성 / CSV not found → generating dummy data

총 샘플 수 / Total samples: 480


,filename,color,level,confidence,level_str
0,scan_dummy_0000.png,Y,0,확실,0
1,scan_dummy_0001.png,Y,0,확실,0
2,scan_dummy_0002.png,Y,0,확실,0
3,scan_dummy_0003.png,Y,0,확실,0
4,scan_dummy_0004.png,Y,0,불확실,0


---
## Cell 3 · 전처리 & Dataset / Preprocessing & Dataset

06번과 동일한 폴더 구조: `labeled/{color}/{level}/{filename}`  
Same folder structure as 06: `labeled/{color}/{level}/{filename}`

In [5]:
# ── 전처리 변환 / Preprocessing transform ────────────────────────────
# 06_embedding_viz.ipynb 와 동일한 transform 사용
# Same transform as 06_embedding_viz.ipynb
# S2 모듈화 이후: preprocessing.py 의 SSOT 전처리로 교체 예정
inference_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # 단일채널→3채널 복제 (PRD §6.8.4)
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],   # ImageNet 기준 (S1 임시)
    ),
])


class PatchDataset(Dataset):
    """
    색상 패치 이미지 Dataset. (06번 PatchDataset 과 동일 구조)
    Color patch image Dataset. (Same structure as 06_embedding_viz PatchDataset)

    폴더 구조 / Folder structure:
        data_set/labeled/{color}/{level}/*.png
    """

    def __init__(self, df: pd.DataFrame, patch_dir: Path, transform):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = Path(patch_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        color = row['color']
        fname = row['filename']
        level = row['level']

        # data_set/labeled/{color}/{level}/{filename} 경로 구성
        img_path = self.patch_dir / color / str(level) / fname

        if img_path.exists():
            img = Image.open(img_path).convert('RGB')
        else:
            # 이미지 미존재 시 더미 이미지 (파이프라인 검증용)
            img = Image.fromarray(
                np.random.randint(100, 200, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
            )

        return self.transform(img), int(level), fname   # tensor, label, filename


print('📐  PatchDataset 정의 완료 / PatchDataset defined')

📐  PatchDataset 정의 완료 / PatchDataset defined


---
## Cell 4 · 모델 로드 / Model Loading

In [6]:
def build_classifier(
    backbone_name : str,
    num_classes   : int,
    checkpoint,
    device        : torch.device,
) -> nn.Module:
    """
    Phase 2 분류기를 생성하고 체크포인트를 로드한다.
    Builds Phase 2 classifier and loads checkpoint.

    PRD §3.0:
        체크포인트 있으면 해당 weights 로드 / Load checkpoint if available
        없으면 ImageNet pretrained 또는 랜덤 가중치 사용
    """
    if backbone_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif backbone_name == 'resnet18':
        model = models.resnet18(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif backbone_name == 'resnet34':
        model = models.resnet34(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f'지원하지 않는 backbone / Unsupported backbone: {backbone_name}')

    if checkpoint is not None and Path(str(checkpoint)).exists():
        # map_location 으로 Windows / macOS 모두 CPU 호환 로드 보장
        state = torch.load(str(checkpoint), map_location='cpu')
        # 순수 state_dict 또는 래핑된 dict 모두 지원
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        model.load_state_dict(state, strict=False)
        print(f'✅  체크포인트 로드 / Checkpoint loaded: {checkpoint}')
    else:
        if checkpoint is not None:
            print(f'⚠️   체크포인트 미존재 → 랜덤 가중치 / Not found → random weights: {checkpoint}')
        else:
            print('ℹ️   체크포인트 미설정 → 랜덤 가중치 (프로토타입 모드)')
            print('ℹ️   No checkpoint set → random weights (prototype mode)')

    model = model.to(device)
    model.eval()   # 평가 모드 / Evaluation mode
    return model


model = build_classifier(
    backbone_name = BACKBONE_NAME,
    num_classes   = NUM_LEVELS,
    checkpoint    = CHECKPOINT,
    device        = DEVICE,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'🧠  Model on device / 디바이스: {DEVICE}')
print(f'    Total parameters / 총 파라미터 수: {total_params:,}')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\hamha/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 53.2MB/s]


ℹ️   체크포인트 미설정 → 랜덤 가중치 (프로토타입 모드)
ℹ️   No checkpoint set → random weights (prototype mode)
🧠  Model on device / 디바이스: cuda
    Total parameters / 총 파라미터 수: 4,015,234


---
## Cell 5 · 추론 실행 / Run Inference

In [7]:
@torch.no_grad()    # 평가 중 그래디언트 비활성화 / Disable gradients during eval
def run_inference(
    model  : nn.Module,
    loader : DataLoader,
    device : torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """
    DataLoader 전체에 대해 모델 추론을 실행합니다.
    Runs model inference over the entire DataLoader.

    Returns:
        y_true      : 정답 라벨 (N,)
        y_pred      : 예측 라벨 (N,)
        confidences : 최대 softmax 신뢰도 (N,)
        filenames   : 이미지 파일명 리스트
    """
    model.eval()
    all_true, all_pred, all_conf, all_fnames = [], [], [], []

    for batch_imgs, batch_labels, batch_fnames in loader:
        batch_imgs  = batch_imgs.to(device, non_blocking=True)
        logits      = model(batch_imgs)              # (B, NUM_LEVELS)
        probs       = torch.softmax(logits, dim=1)   # (B, NUM_LEVELS)
        conf, preds = probs.max(dim=1)               # max class & confidence

        all_true.extend(batch_labels.numpy())
        all_pred.extend(preds.cpu().numpy())
        all_conf.extend(conf.cpu().numpy())
        all_fnames.extend(batch_fnames)

    return (
        np.array(all_true),
        np.array(all_pred),
        np.array(all_conf),
        all_fnames,
    )


# ── 색상별 DataLoader 생성 후 추론 실행 / Build per-color DataLoaders → inference
results: Dict[str, dict] = {}    # 색상별 결과 저장 / Store per-color results

for color in CHANNELS:
    df_color = df_labels[df_labels['color'] == color].reset_index(drop=True)

    ds = PatchDataset(
        df        = df_color,
        patch_dir = LABELED_DIR,    # data_set/labeled/
        transform = inference_transform,
    )
    loader = DataLoader(
        ds,
        batch_size  = BATCH_SIZE,
        shuffle     = False,        # 순서 유지 / Keep order
        num_workers = 0,            # 0 = single-process (Windows/macOS 모두 안전)
        pin_memory  = (DEVICE.type == 'cuda'),
    )

    y_true, y_pred, confs, fnames = run_inference(model, loader, DEVICE)
    results[color] = {
        'y_true'      : y_true,
        'y_pred'      : y_pred,
        'confidences' : confs,
        'filenames'   : fnames,
    }
    acc = accuracy_score(y_true, y_pred)
    print(f'  [{color}] {len(y_true):4d} samples | Accuracy: {acc:.4f}')

print('\n✅  전체 채널 추론 완료 / Inference complete for all channels')

  [Y]  120 samples | Accuracy: 0.1667
  [M]  120 samples | Accuracy: 0.1667
  [C]  120 samples | Accuracy: 0.1667
  [K]  120 samples | Accuracy: 0.1667

✅  전체 채널 추론 완료 / Inference complete for all channels


---
## Cell 6 · 지표 계산 / Metrics Computation

In [8]:
def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    num_classes: int = NUM_LEVELS) -> dict:
    """
    분류 및 순서형 지표를 계산합니다.
    Computes classification and ordinal metrics.
    """
    labels   = list(range(num_classes))
    accuracy = accuracy_score(y_true, y_pred)
    prec     = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    rec      = recall_score   (y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1       = f1_score       (y_true, y_pred, labels=labels, average=None, zero_division=0)
    macro_f1 = f1_score       (y_true, y_pred, average='macro', zero_division=0)

    # MAE: Level을 순서형 정수로 취급 (PRD §1.4)
    mae = float(np.mean(np.abs(y_true.astype(float) - y_pred.astype(float))))

    per_class = [
        {'level': i, 'precision': prec[i], 'recall': rec[i], 'f1': f1[i]}
        for i in labels
    ]
    return {'accuracy': accuracy, 'macro_f1': macro_f1, 'mae': mae, 'per_class': per_class}


# 색상별 지표 / Per-color metrics
metrics: Dict[str, dict] = {}
for color in CHANNELS:
    metrics[color] = compute_metrics(results[color]['y_true'], results[color]['y_pred'])

# 전체 통합 지표 / Overall combined metrics
all_true_combined = np.concatenate([results[c]['y_true'] for c in CHANNELS])
all_pred_combined = np.concatenate([results[c]['y_pred'] for c in CHANNELS])
metrics['overall'] = compute_metrics(all_true_combined, all_pred_combined)

# ── 요약 출력 / Summary print ─────────────────────────────────────────
print('\n=== Performance Summary / 성능 요약 ===')
header = f"{'Channel':>10}  {'Accuracy':>10}  {'Macro F1':>10}  {'MAE':>8}  {'Acc':>4}  {'F1':>4}  {'MAE':>4}"
print(header)
print('-' * len(header))

for ch in CHANNELS + ['overall']:
    m       = metrics[ch]
    tgt_acc = TARGET_OVERALL_ACC if ch == 'overall' else TARGET_PER_COLOR_ACC
    print(
        f"{ch:>10}  {m['accuracy']:>10.4f}  {m['macro_f1']:>10.4f}  {m['mae']:>8.4f}  "
        f"{'✅' if m['accuracy'] >= tgt_acc else '❌':>4}  "
        f"{'✅' if m['macro_f1'] >= TARGET_PER_CLASS_F1 else '❌':>4}  "
        f"{'✅' if m['mae'] <= TARGET_MAE else '❌':>4}"
    )

print()
print(f'Targets (PRD §1.4):')
print(f'  Overall Accuracy ≥ {TARGET_OVERALL_ACC:.0%}')
print(f'  Per-color Acc    ≥ {TARGET_PER_COLOR_ACC:.0%}')
print(f'  Per-class F1     ≥ {TARGET_PER_CLASS_F1:.2f}')
print(f'  MAE              ≤ {TARGET_MAE:.2f}')

print('\n=== Per-Class Performance (Overall) / 클래스별 성능 (전체) ===')
for pc in metrics['overall']['per_class']:
    flag = '✅' if pc['f1'] >= TARGET_PER_CLASS_F1 else '❌'
    print(f"  Level {pc['level']}  Prec={pc['precision']:.4f}  Recall={pc['recall']:.4f}  F1={pc['f1']:.4f}  {flag}")


=== Performance Summary / 성능 요약 ===
   Channel    Accuracy    Macro F1       MAE   Acc    F1   MAE
--------------------------------------------------------------
         Y      0.1667      0.0476    1.8333     ❌     ❌     ❌
         M      0.1667      0.0476    1.8333     ❌     ❌     ❌
         C      0.1667      0.0480    1.8583     ❌     ❌     ❌
         K      0.1667      0.0476    1.8333     ❌     ❌     ❌
   overall      0.1667      0.0477    1.8396     ❌     ❌     ❌

Targets (PRD §1.4):
  Overall Accuracy ≥ 90%
  Per-color Acc    ≥ 85%
  Per-class F1     ≥ 0.80
  MAE              ≤ 0.50

=== Per-Class Performance (Overall) / 클래스별 성능 (전체) ===
  Level 0  Prec=0.0000  Recall=0.0000  F1=0.0000  ❌
  Level 1  Prec=0.1670  Recall=1.0000  F1=0.2862  ❌
  Level 2  Prec=0.0000  Recall=0.0000  F1=0.0000  ❌
  Level 3  Prec=0.0000  Recall=0.0000  F1=0.0000  ❌
  Level 4  Prec=0.0000  Recall=0.0000  F1=0.0000  ❌
  Level 5  Prec=0.0000  Recall=0.0000  F1=0.0000  ❌


---
## Cell 7 · 혼동 행렬 / Confusion Matrix (Plotly HTML)

> `fig.show()` **완전 제거** — HTML 저장 후 `open_in_browser()` 사용  
> `fig.show()` **completely removed** — HTML saved, opened via `open_in_browser()`

In [9]:
def plot_confusion_matrix(
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    title       : str,
    normalize   : bool = True,
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    6×6 혼동 행렬 Plotly 히트맵.
    6×6 confusion matrix as a Plotly heatmap.

    Note:
        fig.show() 제거됨 — macOS VS Code Jupyter 오류 방지
        fig.show() removed — prevents macOS VS Code Jupyter errors
    """
    labels      = list(range(NUM_LEVELS))
    level_names = [f'Level {i}' for i in labels]
    cm          = confusion_matrix(y_true, y_pred, labels=labels)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        z        = np.where(row_sums > 0, cm / row_sums, 0.0)
        z_text   = [[f'{v:.2f}' for v in row] for row in z]
        cbar_title = 'Proportion'
        vmin, vmax = 0.0, 1.0
    else:
        z        = cm.astype(float)
        z_text   = [[str(v) for v in row] for row in cm]
        cbar_title = 'Count'
        vmin, vmax = None, None

    # y축 반전 → Level 0 이 상단 표시 / Reverse y-axis so Level 0 is at top
    z_flip      = z[::-1]
    z_text_flip = z_text[::-1]
    y_labels    = level_names[::-1]

    fig = go.Figure(go.Heatmap(
        z             = z_flip,
        x             = level_names,
        y             = y_labels,
        text          = z_text_flip,
        texttemplate  = '%{text}',
        colorscale    = 'Blues',
        zmin          = vmin,
        zmax          = vmax,
        colorbar      = dict(title=cbar_title),
        hovertemplate = (
            'Predicted / 예측: %{x}<br>'
            'True / 실제: %{y}<br>'
            'Value: %{text}<extra></extra>'
        ),
    ))
    fig.update_layout(
        title       = dict(text=title, font=dict(size=16)),
        xaxis_title = 'Predicted Level / 예측 레벨',
        yaxis_title = 'True Level / 실제 레벨',
        font        = dict(family='Segoe UI', size=12),
        template    = 'plotly_dark',
        width=620, height=530,
        margin      = dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


# ── 색상별 + 전체 혼동 행렬 생성 / Generate per-color + overall CMs ──
# ⚠️ fig.show() 완전 제거 / fig.show() completely removed
for ch in CHANNELS + ['overall']:
    yt  = results[ch]['y_true'] if ch != 'overall' else all_true_combined
    yp  = results[ch]['y_pred'] if ch != 'overall' else all_pred_combined
    acc = metrics[ch]['accuracy']

    html_path = str(OUTPUT_DIR / f'cm_{ch}.html')
    plot_confusion_matrix(
        yt, yp,
        title       = f'[{ch}] Confusion Matrix — Acc={acc:.4f} (Row-Normalized / 행 정규화)',
        normalize   = True,
        output_path = html_path,
    )
    # 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
    open_in_browser(html_path)

print('\n✅  모든 혼동 행렬 저장 완료 / All confusion matrices saved')

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\cm_Y.html
[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\cm_M.html
[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\cm_C.html
[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\cm_K.html
[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\cm_overall.html

✅  모든 혼동 행렬 저장 완료 / All confusion matrices saved


---
## Cell 8 · 평가 대시보드 / Evaluation Dashboard (Gauge + Bar)

06번 `plot_metrics_dashboard` 와 동일한 Gauge + Bar 구성  
Same Gauge + Bar pattern as 06_embedding_viz `plot_metrics_dashboard`

In [23]:

def plot_eval_dashboard(
    metrics     : Dict[str, dict],
    channels    : List[str],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    Accuracy / Macro F1 / MAE 를 Gauge + Bar 로 시각화.
    Visualizes Accuracy / Macro F1 / MAE as Gauge + Bar charts.

    Note:
        fig.show() 제거됨 / fig.show() removed
    """
    m_all = metrics['overall']

    fig = make_subplots(
        rows=2, cols=3,
        specs=[
            [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
            [{'type': 'bar', 'colspan': 3}, None, None],
        ],
        subplot_titles=[
            'Overall Accuracy / 전체 정확도',
            'Overall Macro F1',
            'Overall MAE (낮을수록 좋음 / Lower is better)',
            'Per-Color Accuracy / 색상별 정확도',
        ],
        vertical_spacing=0.18,
    )

    # Gauge: Overall Accuracy
    fig.add_trace(go.Indicator(
        mode  = 'gauge+number',
        value = round(m_all['accuracy'] * 100, 2),
        number= {'suffix': '%', 'font': {'size': 32}},
        gauge = {
            'axis'     : {'range': [0, 100]},
            'bar'      : {'color': '#50e3c2'},
            'threshold': {'line': {'color': '#ff7aa2', 'width': 3},
                          'value': TARGET_OVERALL_ACC * 100},
            'bgcolor'  : '#0b1220',
        },
        title = {'text': f'Target / 목표 ≥ {TARGET_OVERALL_ACC:.0%}'},
    ), row=1, col=1)

    # Gauge: Macro F1
    fig.add_trace(go.Indicator(
        mode  = 'gauge+number',
        value = round(m_all['macro_f1'], 4),
        number= {'font': {'size': 32}},
        gauge = {
            'axis'     : {'range': [0, 1]},
            'bar'      : {'color': '#66d9ff'},
            'threshold': {'line': {'color': '#ff7aa2', 'width': 3},
                          'value': TARGET_PER_CLASS_F1},
            'bgcolor'  : '#0b1220',
        },
        title = {'text': f'Target / 목표 ≥ {TARGET_PER_CLASS_F1:.2f}'},
    ), row=1, col=2)

    # Gauge: MAE (낮을수록 좋음)
    fig.add_trace(go.Indicator(
        mode  = 'gauge+number',
        value = round(m_all['mae'], 4),
        number= {'font': {'size': 32}},
        gauge = {
            'axis'     : {'range': [0, 3]},
            'bar'      : {'color': '#c792ea'},
            'threshold': {'line': {'color': '#ffb347', 'width': 3},
                          'value': TARGET_MAE},
            'bgcolor'  : '#0b1220',
        },
        title = {'text': f'Target / 목표 ≤ {TARGET_MAE:.2f}'},
    ), row=1, col=3)

    # Bar: 색상별 Accuracy
    bar_colors = [CMYK_COLORS[c] for c in channels]
    acc_vals   = [metrics[c]['accuracy'] * 100 for c in channels]
    fig.add_trace(go.Bar(
        x            = channels,
        y            = acc_vals,
        marker_color = bar_colors,
        text         = [f'{v:.2f}%' for v in acc_vals],
        textposition = 'outside',
        name         = 'Accuracy',
    ), row=2, col=1)

    
    # 목표선 / Target line

    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(channels) - 0.5,
        y0=TARGET_PER_COLOR_ACC * 100,
        y1=TARGET_PER_COLOR_ACC * 100,
        line=dict(color='#ff7aa2', dash='dash'),
        xref='x',
        yref='y',
        row=2,
        col=1,
    )

# 텍스트 따로 추가
    fig.add_annotation(
        x=channels[-1],
        y=TARGET_PER_COLOR_ACC * 100,
        text=f'Target {TARGET_PER_COLOR_ACC:.0%}',
        showarrow=False,
        yshift=10,
        row=2,
        col=1,
    )

    fig.update_layout(
        title      = dict(text='Grayspot Evaluation Dashboard / 평가 대시보드', font=dict(size=18)),
        template   = 'plotly_dark',
        font       = dict(family='Segoe UI', size=12),
        height     = 750,
        showlegend = False,
        margin     = dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


dashboard_path = str(OUTPUT_DIR / 'eval_dashboard.html')
plot_eval_dashboard(metrics, CHANNELS, output_path=dashboard_path)
# 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
open_in_browser(dashboard_path)

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\eval_dashboard.html


---
## Cell 9 · 클래스별 F1 차트 / Per-Class F1 Chart

In [11]:
def plot_per_class_metrics(
    metrics     : Dict[str, dict],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    클래스별 Precision / Recall / F1 막대 차트.
    Per-class Precision / Recall / F1 bar chart.

    Note:
        fig.show() 제거됨 / fig.show() removed
    """
    pc      = metrics['overall']['per_class']
    levels  = [f"Level {d['level']}" for d in pc]

    fig = go.Figure()
    fig.add_trace(go.Bar(name='F1',        x=levels, y=[d['f1']        for d in pc], marker_color='#1976D2'))
    fig.add_trace(go.Bar(name='Precision', x=levels, y=[d['precision'] for d in pc], marker_color='#388E3C'))
    fig.add_trace(go.Bar(name='Recall',    x=levels, y=[d['recall']    for d in pc], marker_color='#F57C00'))

    fig.add_hline(
        y          = TARGET_PER_CLASS_F1,
        line_dash  = 'dash',
        line_color = '#ff7aa2',
        annotation_text = f'F1 Target / 목표 ≥ {TARGET_PER_CLASS_F1:.2f}',
    )

    fig.update_layout(
        title    = dict(text='Per-Class Metrics (Overall) / 클래스별 지표 (전체)', font=dict(size=16)),
        barmode  = 'group',
        yaxis    = dict(range=[0, 1.15], title='Score'),
        xaxis    = dict(title='Level'),
        template = 'plotly_dark',
        font     = dict(family='Segoe UI', size=12),
        legend   = dict(orientation='h', y=1.1),
        height   = 480,
        margin   = dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


per_class_path = str(OUTPUT_DIR / 'per_class_metrics.html')
plot_per_class_metrics(metrics, output_path=per_class_path)
# 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
open_in_browser(per_class_path)

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\per_class_metrics.html


---
## Cell 10 · MAE 히트맵 / MAE Heatmap

In [12]:
def plot_mae_heatmap(
    results     : Dict[str, dict],
    channels    : List[str],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    (색상 × 실제 레벨) 조합별 MAE 히트맵.
    MAE heatmap by (color × true level) combination.

    Note:
        fig.show() 제거됨 / fig.show() removed
    """
    level_names  = [f'Level {i}' for i in range(NUM_LEVELS)]
    mae_matrix   = np.full((len(channels), NUM_LEVELS), np.nan)
    count_matrix = np.zeros((len(channels), NUM_LEVELS), dtype=int)

    for ci, color in enumerate(channels):
        yt = results[color]['y_true']
        yp = results[color]['y_pred']
        for lv in range(NUM_LEVELS):
            mask = yt == lv
            if mask.sum() > 0:
                mae_matrix[ci, lv]   = float(np.mean(np.abs(yt[mask] - yp[mask])))
                count_matrix[ci, lv] = int(mask.sum())

    annot_text = [
        [
            f'{mae_matrix[r, c]:.2f}\n(n={count_matrix[r, c]})'
            if not np.isnan(mae_matrix[r, c]) else 'N/A'
            for c in range(NUM_LEVELS)
        ]
        for r in range(len(channels))
    ]

    fig = go.Figure(go.Heatmap(
        z             = mae_matrix,
        x             = level_names,
        y             = channels,
        text          = annot_text,
        texttemplate  = '%{text}',
        colorscale    = 'YlOrRd',
        zmin          = 0,
        zmax          = 2.0,
        colorbar      = dict(title='MAE'),
        hovertemplate = 'Color: %{y}<br>Level: %{x}<br>MAE: %{z:.3f}<extra></extra>',
    ))
    fig.update_layout(
        title    = dict(
            text = f'MAE per (Color × True Level) — Target / 목표 ≤ {TARGET_MAE}',
            font = dict(size=16),
        ),
        xaxis    = dict(title='True Level / 실제 레벨'),
        yaxis    = dict(title='Color Channel / 색상 채널'),
        template = 'plotly_dark',
        font     = dict(family='Segoe UI', size=12),
        height   = 360,
        margin   = dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


mae_path = str(OUTPUT_DIR / 'mae_heatmap.html')
plot_mae_heatmap(results, CHANNELS, output_path=mae_path)
# 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
open_in_browser(mae_path)

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\mae_heatmap.html


---
## Cell 11 · 오분류 샘플 / Misclassified Samples

06번 `mismatch_samples.csv` 와 동일한 포맷으로 저장  
Saved in the same format as 06_embedding_viz `mismatch_samples.csv`

In [13]:
# ── 오분류 샘플 수집 / Collect misclassified samples ──────────────────
misclassified_records: List[dict] = []

for color in CHANNELS:
    yt    = results[color]['y_true']
    yp    = results[color]['y_pred']
    confs = results[color]['confidences']
    fnames= results[color]['filenames']

    for i in np.where(yt != yp)[0]:
        misclassified_records.append({
            'filename'   : fnames[i],
            'color'      : color,
            'true_level' : int(yt[i]),
            'pred_level' : int(yp[i]),
            'confidence' : round(float(confs[i]), 4),
            'correct'    : False,
            'error_gap'  : int(abs(yt[i] - yp[i])),
        })

df_miss = pd.DataFrame(misclassified_records)
if len(df_miss) > 0:
    df_miss = df_miss.sort_values(['error_gap', 'confidence'], ascending=[False, True])
    print(f'⚠️   오분류 샘플 수 / Misclassified: {len(df_miss)}')
    print(f'    최대 오류 간격 / Max error gap: {df_miss["error_gap"].max()}')
    print('\n[색상별 오분류 수 / Per-color mismatch]')
    print(df_miss.groupby('color').size().to_string())
else:
    print('✅  오분류 없음 / No misclassifications (prototype mode with random weights)')

# ── 저장 (06번과 동일 포맷) / Save (same format as 06_embedding_viz) ──
# UTF-8 BOM: Windows Excel 한글 깨짐 방지 / Prevents Korean garbling in Windows Excel
miss_path = OUTPUT_DIR / 'misclassified_samples.csv'
df_miss.to_csv(miss_path, index=False, encoding='utf-8-sig')
print(f'\n[저장 / Saved] {miss_path}')
df_miss.head(10)

⚠️   오분류 샘플 수 / Misclassified: 400
    최대 오류 간격 / Max error gap: 4

[색상별 오분류 수 / Per-color mismatch]
color
C    100
K    100
M    100
Y    100

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\misclassified_samples.csv


,filename,color,true_level,pred_level,confidence,correct,error_gap
384,scan_dummy_0464.png,K,5,1,0.1857,False,4
396,scan_dummy_0476.png,K,5,1,0.1919,False,4
182,scan_dummy_0222.png,M,5,1,0.1920,False,4
288,scan_dummy_0348.png,C,5,1,0.1932,False,4
190,scan_dummy_0230.png,M,5,1,0.1938,False,4
198,scan_dummy_0238.png,M,5,1,0.1941,False,4
287,scan_dummy_0347.png,C,5,1,0.1949,False,4
199,scan_dummy_0239.png,M,5,1,0.1953,False,4
289,scan_dummy_0349.png,C,5,1,0.1953,False,4
83,scan_dummy_0103.png,Y,5,1,0.1956,False,4


In [14]:
def plot_mismatch_scatter(
    df_miss     : pd.DataFrame,
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    오분류 샘플 scatter plot (color=채널, size=error_gap).
    Misclassified samples scatter: color=channel, size=error_gap.

    Note:
        fig.show() 제거됨 / fig.show() removed
    """
    if len(df_miss) == 0:
        print('ℹ️   오분류 없음 — 시각화 건너뜀 / No misclassified samples — skipping')
        return go.Figure()

    fig = px.scatter(
        df_miss,
        x                  = 'true_level',
        y                  = 'pred_level',
        color              = 'color',
        color_discrete_map = CMYK_COLORS,
        size               = 'error_gap',
        size_max           = 20,
        hover_data         = ['filename', 'color', 'true_level', 'pred_level', 'confidence', 'error_gap'],
        title              = 'Misclassified Samples — True vs Predicted Level / 오분류 샘플',
        labels             = {
            'true_level': 'True Level / 실제 레벨',
            'pred_level': 'Predicted Level / 예측 레벨',
            'color'     : 'CMYK Channel',
        },
        template           = 'plotly_dark',
        width=700, height=550,
    )

    # 대각선 (정답 경계) / Diagonal (correct boundary)
    fig.add_trace(go.Scatter(
        x=[0, NUM_LEVELS - 1], y=[0, NUM_LEVELS - 1],
        mode='lines',
        line=dict(color='gray', dash='dash', width=1),
        name='Correct boundary / 정답 경계',
        showlegend=True,
    ))

    fig.update_layout(
        font   = dict(family='Segoe UI', size=12),
        margin = dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


scatter_path = str(OUTPUT_DIR / 'misclassified_scatter.html')
plot_mismatch_scatter(df_miss, output_path=scatter_path)
# 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
open_in_browser(scatter_path)

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\misclassified_scatter.html


---
## Cell 12 · 신뢰도 분포 / Confidence Distribution (PRD §14.2)

In [15]:
def plot_confidence_distribution(
    results     : Dict[str, dict],
    channels    : List[str],
    output_path : Optional[str] = None,
) -> go.Figure:
    """
    색상별 신뢰도 히스토그램 (정답 vs 오답).
    PRD §14.2 임계값 수직선 표시.
    Per-color confidence histogram (correct vs wrong).
    Shows PRD §14.2 threshold vertical lines.

    Note:
        fig.show() 제거됨 / fig.show() removed
    """
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles      = [f'[{c}]' for c in channels],
        horizontal_spacing  = 0.10,
        vertical_spacing    = 0.18,
    )

    bins = dict(start=0, end=1, size=0.04)

    for i, color in enumerate(channels):
        r  = i // 2 + 1
        c  = i % 2  + 1
        yt = results[color]['y_true']
        yp = results[color]['y_pred']
        cf = results[color]['confidences']

        # 정답 히스토그램 / Correct histogram
        fig.add_trace(go.Histogram(
            x          = cf[yt == yp],
            xbins      = bins,
            name       = '정답 / Correct',
            marker_color = '#4fc3f7',
            opacity    = 0.70,
            showlegend = (i == 0),
            legendgroup = 'correct',
        ), row=r, col=c)

        # 오답 히스토그램 / Wrong histogram
        fig.add_trace(go.Histogram(
            x          = cf[yt != yp],
            xbins      = bins,
            name       = '오답 / Wrong',
            marker_color = '#ef5350',
            opacity    = 0.70,
            showlegend = (i == 0),
            legendgroup = 'wrong',
        ), row=r, col=c)

        # 임계값 수직선 / Threshold vertical lines
        for thresh, col_name in [
            (CONF_THRESH_AUTO,   'green'),
            (CONF_THRESH_WARN,   'orange'),
            (CONF_THRESH_MANUAL, 'red'),
        ]:
            fig.add_vline(
                x          = thresh,
                line_dash  = 'dash',
                line_color = col_name,
                line_width = 1.5,
                row=r, col=c,
            )

    fig.update_layout(
        title    = dict(
            text = 'Confidence Distribution / 신뢰도 분포 — Correct vs Wrong',
            font = dict(size=16),
        ),
        barmode  = 'overlay',
        template = 'plotly_dark',
        font     = dict(family='Segoe UI', size=12),
        height   = 650,
        margin   = dict(l=40, r=40, t=80, b=40),
    )

    if output_path:
        fig.write_html(output_path, include_plotlyjs='cdn')
        print(f'[저장 / Saved] {output_path}')

    return fig


conf_path = str(OUTPUT_DIR / 'confidence_distribution.html')
plot_confidence_distribution(results, CHANNELS, output_path=conf_path)
# 저장된 파일을 브라우저로 직접 열기 / Open saved file in browser
open_in_browser(conf_path)

# ── 커버리지 테이블 출력 / Coverage table ─────────────────────────────
all_confs = np.concatenate([results[c]['confidences'] for c in CHANNELS])
total     = len(all_confs)

print('\n=== Confidence-Driven Coverage (PRD §14.2) / 신뢰도 기반 커버리지 ===')
print(f"{'Policy':30s} {'Threshold':>12s} {'Samples':>10s} {'Coverage':>10s}")
print('-' * 68)
for name, thresh in [
    ('Auto judgment / 자동 판정',     CONF_THRESH_AUTO),
    ('Warn + auto  / 경고 포함 자동',  CONF_THRESH_WARN),
    ('Manual queue / 수동 검수 대기',  CONF_THRESH_MANUAL),
]:
    n   = int((all_confs >= thresh).sum())
    pct = n / total * 100
    print(f"{name:30s} {'≥ ' + str(thresh):>12s} {n:>10d} {pct:>9.1f}%")
print(f"{'Total / 전체':30s} {'—':>12s} {total:>10d} {'100.0':>9s}%")

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\confidence_distribution.html

=== Confidence-Driven Coverage (PRD §14.2) / 신뢰도 기반 커버리지 ===
Policy                            Threshold    Samples   Coverage
--------------------------------------------------------------------
Auto judgment / 자동 판정                 ≥ 0.8          0       0.0%
Warn + auto  / 경고 포함 자동               ≥ 0.5          0       0.0%
Manual queue / 수동 검수 대기               ≥ 0.3          0       0.0%
Total / 전체                                —        480     100.0%


---
## Cell 13 · 결과 내보내기 / Export Results (CSV + JSON)

In [17]:
# ── evaluation_results.csv (PRD §8.2.2 포맷) ──────────────────────────
eval_rows = []
for color in CHANNELS:
    yt    = results[color]['y_true']
    yp    = results[color]['y_pred']
    confs = results[color]['confidences']
    fnames= results[color]['filenames']
    for i in range(len(yt)):
        eval_rows.append({
            'filename'   : fnames[i],
            'color'      : color,
            'true_level' : int(yt[i]),
            'pred_level' : int(yp[i]),
            'confidence' : round(float(confs[i]), 4),
            'correct'    : bool(yt[i] == yp[i]),
            'error_gap'  : int(abs(yt[i] - yp[i])),
        })

df_eval  = pd.DataFrame(eval_rows)
eval_csv = OUTPUT_DIR / 'evaluation_results.csv'
df_eval.to_csv(eval_csv, index=False, encoding='utf-8-sig')
print(f'[저장 / Saved] {eval_csv}  ({len(df_eval)} rows)')


# ── metrics_summary.json (06번과 동일한 JSON 구조) ────────────────────
metrics_export = {
    'meta': {
        'n_samples' : int(len(df_labels)),
        'backbone'  : BACKBONE_NAME,
        'checkpoint': str(CHECKPOINT) if CHECKPOINT else None,
    },
    'targets': {
        'overall_accuracy'  : TARGET_OVERALL_ACC,
        'per_color_accuracy': TARGET_PER_COLOR_ACC,
        'per_class_f1'      : TARGET_PER_CLASS_F1,
        'mae'               : TARGET_MAE,
    },
    'global': {
        'accuracy' : round(metrics['overall']['accuracy'],  4),
        'macro_f1' : round(metrics['overall']['macro_f1'],  4),
        'mae'      : round(metrics['overall']['mae'],        4),
        'acc_pass' : bool(metrics['overall']['accuracy']  >= TARGET_OVERALL_ACC),
        'f1_pass'  : bool(metrics['overall']['macro_f1']  >= TARGET_PER_CLASS_F1),
        'mae_pass' : bool(metrics['overall']['mae']        <= TARGET_MAE),
    },
    'by_color': {
        color: {
            'accuracy': round(metrics[color]['accuracy'],  4),
            'macro_f1': round(metrics[color]['macro_f1'],  4),
            'mae'     : round(metrics[color]['mae'],        4),
            'acc_pass': bool(metrics[color]['accuracy'] >= TARGET_PER_COLOR_ACC),
        }
        for color in CHANNELS
    },
    'per_class_overall': [
        {
            'level'    : pc['level'],
            'precision': round(pc['precision'], 4),
            'recall'   : round(pc['recall'],    4),
            'f1'       : round(pc['f1'],         4),
            'f1_pass'  : bool(pc['f1'] >= TARGET_PER_CLASS_F1),
        }
        for pc in metrics['overall']['per_class']
    ],
}

json_path = OUTPUT_DIR / 'metrics_summary.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_export, f, ensure_ascii=False, indent=2)
print(f'[저장 / Saved] {json_path}')

[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\evaluation_results.csv  (480 rows)
[저장 / Saved] C:\visualcode\CMYK_jin_clone\CMYK_MAIN\data_set\embedding_viz\metrics_summary.json


---
## Cell 14 · Phase 3 피드백 복귀 판단 / Phase 3 Feedback Decision

PRD §3.3.2 피드백 루프 판단 기준  
Implements PRD §3.3.2 feedback-loop decision logic

In [18]:
print('=' * 65)
print('  Phase 3 Feedback Decision / Phase 3 피드백 복귀 판단')
print('=' * 65)

decisions: List[str] = []

# 검사 1: 색상별 정확도 미달 → Phase 0 복귀
# Check 1: Per-color accuracy below threshold → Phase 0
for color in CHANNELS:
    acc = metrics[color]['accuracy']
    if acc < 0.80:
        decisions.append(
            f'  ⮕ [{color}] Accuracy {acc:.3f} < 0.80 → '
            'Phase 0 (표현 재학습 / Retrain representation)'
        )

# 검사 2: 클래스별 F1 < 0.70 → Phase 1 복귀
# Check 2: Per-class F1 below 0.70 → Phase 1
for pc in metrics['overall']['per_class']:
    if pc['f1'] < 0.70:
        decisions.append(
            f"  ⮕ Level {pc['level']} F1={pc['f1']:.3f} < 0.70 → "
            'Phase 1 (레벨 경계 재검토 / Review level boundary)'
        )

# 검사 3: MAE > 0.80 → Phase 0 복귀
# Check 3: MAE above 0.80 → Phase 0
overall_mae = metrics['overall']['mae']
if overall_mae > 0.80:
    decisions.append(
        f'  ⮕ Overall MAE {overall_mae:.3f} > 0.80 → '
        'Phase 0 (표현 학습 재시도 / Representation learning retry)'
    )

# 검사 4: 모든 목표 달성 → Swing 종료
# Check 4: All targets met → terminate
overall_acc  = metrics['overall']['accuracy']
overall_mf1  = metrics['overall']['macro_f1']
all_color_ok = all(metrics[c]['accuracy'] >= TARGET_PER_COLOR_ACC for c in CHANNELS)
all_class_ok = all(pc['f1'] >= TARGET_PER_CLASS_F1 for pc in metrics['overall']['per_class'])
mae_ok       = overall_mae <= TARGET_MAE

if overall_acc >= TARGET_OVERALL_ACC and all_color_ok and all_class_ok and mae_ok:
    print('\n  🎉 모든 목표 달성 → Swing 종료 / All targets met — TERMINATE Swing')
    print(f'     Overall Accuracy : {overall_acc:.4f} ≥ {TARGET_OVERALL_ACC}')
    print(f'     Macro F1         : {overall_mf1:.4f} ≥ {TARGET_PER_CLASS_F1}')
    print(f'     MAE              : {overall_mae:.4f} ≤ {TARGET_MAE}')
elif not decisions:
    print('\n  ✅  심각한 실패 없음 — 학습 계속 또는 다음 사이클')
    print('  ✅  No critical failures — continue training or next cycle')
else:
    print('\n  ⚠️  조치 필요 / Action required:')
    for d in decisions:
        print(d)

print()
print(f'  Overall Accuracy : {overall_acc:.4f}  (target ≥ {TARGET_OVERALL_ACC})')
print(f'  Overall Macro F1 : {overall_mf1:.4f}  (target ≥ {TARGET_PER_CLASS_F1})')
print(f'  Overall MAE      : {overall_mae:.4f}  (target ≤ {TARGET_MAE})')
print('=' * 65)

  Phase 3 Feedback Decision / Phase 3 피드백 복귀 판단

  ⚠️  조치 필요 / Action required:
  ⮕ [Y] Accuracy 0.167 < 0.80 → Phase 0 (표현 재학습 / Retrain representation)
  ⮕ [M] Accuracy 0.167 < 0.80 → Phase 0 (표현 재학습 / Retrain representation)
  ⮕ [C] Accuracy 0.167 < 0.80 → Phase 0 (표현 재학습 / Retrain representation)
  ⮕ [K] Accuracy 0.167 < 0.80 → Phase 0 (표현 재학습 / Retrain representation)
  ⮕ Level 0 F1=0.000 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Level 1 F1=0.286 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Level 2 F1=0.000 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Level 3 F1=0.000 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Level 4 F1=0.000 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Level 5 F1=0.000 < 0.70 → Phase 1 (레벨 경계 재검토 / Review level boundary)
  ⮕ Overall MAE 1.840 > 0.80 → Phase 0 (표현 학습 재시도 / Representation learning retry)

  Overall Accuracy : 0.1667  (target ≥ 0.9)
  Overall Macro F1 : 0.0477  (target ≥ 0.8)
  Overal

---
## Cell 15 · 생성된 산출물 목록 / Generated Outputs

06번 마지막 셀과 동일한 포맷  
Same format as the final cell of 06_embedding_viz

In [19]:
print('=' * 65)
print('Generated Outputs / 생성된 산출물 목록')
print('=' * 65)

output_files = [
    # ── Plotly HTML (브라우저에서 열기) ───────────────────────────────
    ('cm_Y.html',                    'Confusion Matrix [Y] — interactive / 혼동 행렬'),
    ('cm_M.html',                    'Confusion Matrix [M] — interactive / 혼동 행렬'),
    ('cm_C.html',                    'Confusion Matrix [C] — interactive / 혼동 행렬'),
    ('cm_K.html',                    'Confusion Matrix [K] — interactive / 혼동 행렬'),
    ('cm_overall.html',              'Confusion Matrix [Overall] — interactive / 전체'),
    ('eval_dashboard.html',          'Accuracy / F1 / MAE Gauge dashboard / 평가 대시보드'),
    ('per_class_metrics.html',       'Per-class Precision / Recall / F1 / 클래스별 지표'),
    ('mae_heatmap.html',             'MAE heatmap (color × level) / MAE 히트맵'),
    ('misclassified_scatter.html',   'Misclassified samples scatter / 오분류 분포'),
    ('confidence_distribution.html', 'Confidence distribution / 신뢰도 분포'),
    # ── CSV / JSON 데이터 파일 ─────────────────────────────────────────
    ('evaluation_results.csv',       'Per-sample predictions / 샘플별 예측 결과'),
    ('misclassified_samples.csv',    'Misclassified sample list / 오분류 샘플 목록'),
    ('metrics_summary.json',         'Aggregated metrics JSON / 성능 요약 JSON'),
]

for fname, desc in output_files:
    full_path = OUTPUT_DIR / fname
    exists    = '[OK]' if full_path.exists() else '[  ]'
    print(f'  {exists}  {fname:<45} {desc}')

print()
print('Next steps / 다음 단계 (PRD §3.3.2):')
print('  1. Open eval_dashboard.html → check Accuracy / F1 / MAE targets')
print('     eval_dashboard.html 을 열고 목표 달성 여부 확인')
print('  2. Review misclassified_samples.csv by error_gap (descending)')
print('     misclassified_samples.csv 를 error_gap 내림차순으로 검토')
print('  3. Cross-check with 06_embedding_viz mismatch_samples.csv')
print('     06번 mismatch_samples.csv 와 불일치 패턴 교차 확인')
print('  4. Phase 0 복귀 → backbone 재학습 / Phase 1 복귀 → 라벨 재정제')
print('  5. 모든 목표 달성 → 모델 고정 후 Stage 5 Delivery 진행')

Generated Outputs / 생성된 산출물 목록
  [OK]  cm_Y.html                                     Confusion Matrix [Y] — interactive / 혼동 행렬
  [OK]  cm_M.html                                     Confusion Matrix [M] — interactive / 혼동 행렬
  [OK]  cm_C.html                                     Confusion Matrix [C] — interactive / 혼동 행렬
  [OK]  cm_K.html                                     Confusion Matrix [K] — interactive / 혼동 행렬
  [OK]  cm_overall.html                               Confusion Matrix [Overall] — interactive / 전체
  [  ]  eval_dashboard.html                           Accuracy / F1 / MAE Gauge dashboard / 평가 대시보드
  [OK]  per_class_metrics.html                        Per-class Precision / Recall / F1 / 클래스별 지표
  [OK]  mae_heatmap.html                              MAE heatmap (color × level) / MAE 히트맵
  [OK]  misclassified_scatter.html                    Misclassified samples scatter / 오분류 분포
  [OK]  confidence_distribution.html                  Confidence distribution / 신뢰도 분포
  [OK]  eva